In [2]:
# Ячейка 1: Установка библиотек
!pip install ipywidgets pandas plotly

# Ячейка 2: Полный код CRM с документацией
"""
CRM SYSTEM PROTOTYPE FOR RETAIL SALES OPTIMIZATION
===================================================
Автор: Студент
Дата: 19.04.2026
Версия: 2.0

Описание:
    Прототип CRM-системы для розничной торговли. Позволяет управлять клиентской базой,
    учитывать продажи, рассчитывать ключевые показатели эффективности (KPI) и
    визуализировать аналитику.

Функциональность:
    - Просмотр списка клиентов
    - Добавление новых клиентов
    - Добавление продаж с привязкой к клиенту
    - Автоматический расчёт LTV, среднего чека, конверсии, общей выручки
    - Визуализация продаж по категориям товаров
    - Хранение данных в SQLite базе данных

Технологии:
    - Python 3.10+
    - SQLite (встроенная база данных)
    - Pandas (обработка данных)
    - Plotly (визуализация)
    - ipywidgets (интерактивный интерфейс)
"""

# Импорт необходимых библиотек
import pandas as pd  # для работы с таблицами данных (DataFrame)
import sqlite3  # для работы с SQLite базой данных
import plotly.express as px  # для создания интерактивных графиков
from datetime import datetime  # для работы с датами и временем
from typing import Tuple, Dict, List, Optional, Any  # для аннотации типов
import ipywidgets as widgets  # для создания интерактивных элементов (кнопки, меню)
from IPython.display import display, clear_output  # для отображения и очистки вывода
import warnings  # для управления предупреждениями

warnings.filterwarnings('ignore')  # отключаем предупреждения для чистоты вывода


class DatabaseManager:
    """
    Класс для управления базой данных.

    Отвечает за:
        - Подключение к базе данных
        - Создание таблиц
        - Выполнение операций добавления и чтения данных
        - Закрытие соединения

    Атрибуты:
        db_path (str): путь к файлу базы данных
        conn (sqlite3.Connection): объект соединения с БД
        cursor (sqlite3.Cursor): курсор для выполнения запросов
    """

    def __init__(self, db_path: str = 'crm.db') -> None:
        """
        Инициализация менеджера базы данных.

        Параметры:
            db_path (str): путь к файлу базы данных. По умолчанию 'crm.db'

        Возвращает:
            None
        """
        self.db_path: str = db_path  # сохраняем путь к файлу БД
        self.conn: Optional[sqlite3.Connection] = None  # соединение с БД
        self.cursor: Optional[sqlite3.Cursor] = None  # курсор для запросов

        self._connect()  # устанавливаем соединение
        self._create_tables()  # создаём таблицы если их нет
        self._add_test_data()  # добавляем тестовые данные если таблицы пустые

    def _connect(self) -> None:
        """
        Устанавливает соединение с SQLite базой данных.

        Параметры:
            None

        Возвращает:
            None
        """
        self.conn = sqlite3.connect(self.db_path)  # подключаемся к файлу БД
        self.cursor = self.conn.cursor()  # создаём курсор для выполнения запросов

    def _create_tables(self) -> None:
        """
        Создаёт необходимые таблицы в базе данных, если они не существуют.

        Таблицы:
            - clients: хранит информацию о клиентах
            - sales: хранит информацию о продажах

        Параметры:
            None

        Возвращает:
            None
        """
        # Создаём таблицу клиентов
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS clients (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                name TEXT NOT NULL,
                phone TEXT,
                email TEXT,
                registration_date TEXT,
                segment TEXT
            )
        ''')

        # Создаём таблицу продаж
        self.cursor.execute('''
            CREATE TABLE IF NOT EXISTS sales (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                client_id INTEGER,
                sale_date TEXT,
                amount REAL,
                product_category TEXT,
                FOREIGN KEY (client_id) REFERENCES clients (id)
            )
        ''')

        self.conn.commit()  # сохраняем изменения

    def _add_test_data(self) -> None:
        """
        Добавляет тестовые данные в пустую базу данных.

        Если таблица клиентов пуста, добавляет:
            - 4 тестовых клиента
            - 8 тестовых продаж (по 2 на каждого клиента)

        Параметры:
            None

        Возвращает:
            None
        """
        # Проверяем, есть ли данные в таблице клиентов
        self.cursor.execute("SELECT COUNT(*) FROM clients")
        client_count: int = self.cursor.fetchone()[0]

        if client_count == 0:  # если клиентов нет
            # Список тестовых клиентов
            test_clients: List[Tuple[str, str, str, str, str]] = [
                ('Иванов Иван', '+7-999-123-45-67', 'ivan@mail.ru', '2025-01-15', 'VIP'),
                ('Петрова Мария', '+7-999-234-56-78', 'maria@mail.ru', '2025-01-20', 'Обычный'),
                ('Сидоров Алексей', '+7-999-345-67-89', 'alex@mail.ru', '2025-02-01', 'VIP'),
                ('Козлова Елена', '+7-999-456-78-90', 'elena@mail.ru', '2025-02-10', 'Обычный'),
            ]

            # Добавляем каждого клиента
            for client in test_clients:
                self.cursor.execute('''
                    INSERT INTO clients (name, phone, email, registration_date, segment)
                    VALUES (?, ?, ?, ?, ?)
                ''', client)

            # Добавляем тестовые продажи для каждого клиента
            for client_id in range(1, 5):  # для клиентов с id от 1 до 4
                # Первая продажа: Электроника на 5000 рублей
                self.cursor.execute('''
                    INSERT INTO sales (client_id, sale_date, amount, product_category)
                    VALUES (?, ?, ?, ?)
                ''', (client_id, '2025-03-15', 5000.0, 'Электроника'))

                # Вторая продажа: Одежда на 3000 рублей
                self.cursor.execute('''
                    INSERT INTO sales (client_id, sale_date, amount, product_category)
                    VALUES (?, ?, ?, ?)
                ''', (client_id, '2025-03-20', 3000.0, 'Одежда'))

            self.conn.commit()  # сохраняем изменения

    def add_client(self, name: str, phone: str, email: str, segment: str) -> bool:
        """
        Добавляет нового клиента в базу данных.

        Параметры:
            name (str): ФИО клиента
            phone (str): Номер телефона клиента
            email (str): Адрес электронной почты
            segment (str): Сегмент клиента (VIP, Обычный, Новый)

        Возвращает:
            bool: True если добавление успешно, False если произошла ошибка
        """
        try:
            # Текущая дата в формате ГГГГ-ММ-ДД
            registration_date: str = datetime.now().strftime('%Y-%m-%d')

            # Выполняем запрос на добавление клиента
            self.cursor.execute('''
                INSERT INTO clients (name, phone, email, registration_date, segment)
                VALUES (?, ?, ?, ?, ?)
            ''', (name, phone, email, registration_date, segment))

            self.conn.commit()  # сохраняем изменения
            return True  # успешное добавление

        except Exception as e:
            print(f"Ошибка при добавлении клиента: {e}")
            return False  # ошибка при добавлении

    def add_sale(self, client_id: int, amount: float, category: str) -> bool:
        """
        Добавляет новую продажу в базу данных.

        Параметры:
            client_id (int): ID клиента, совершившего покупку
            amount (float): Сумма покупки в рублях
            category (str): Категория товара

        Возвращает:
            bool: True если добавление успешно, False если произошла ошибка
        """
        try:
            # Текущая дата в формате ГГГГ-ММ-ДД
            sale_date: str = datetime.now().strftime('%Y-%m-%d')

            # Выполняем запрос на добавление продажи
            self.cursor.execute('''
                INSERT INTO sales (client_id, sale_date, amount, product_category)
                VALUES (?, ?, ?, ?)
            ''', (client_id, sale_date, amount, category))

            self.conn.commit()  # сохраняем изменения
            return True  # успешное добавление

        except Exception as e:
            print(f"Ошибка при добавлении продажи: {e}")
            return False  # ошибка при добавлении

    def get_clients(self) -> pd.DataFrame:
        """
        Получает всех клиентов из базы данных.

        Параметры:
            None

        Возвращает:
            pd.DataFrame: DataFrame с данными всех клиентов
        """
        query: str = "SELECT * FROM clients"
        return pd.read_sql_query(query, self.conn)

    def get_sales(self) -> pd.DataFrame:
        """
        Получает все продажи из базы данных.

        Параметры:
            None

        Возвращает:
            pd.DataFrame: DataFrame с данными всех продаж
        """
        query: str = "SELECT * FROM sales"
        return pd.read_sql_query(query, self.conn)

    def get_clients_with_sales(self) -> pd.DataFrame:
        """
        Получает клиентов с суммой и количеством их покупок.

        Параметры:
            None

        Возвращает:
            pd.DataFrame: DataFrame с клиентами и их покупательской активностью
        """
        clients: pd.DataFrame = self.get_clients()
        sales: pd.DataFrame = self.get_sales()

        if len(sales) > 0:
            # Группируем продажи по клиентам
            sales_summary: pd.DataFrame = sales.groupby('client_id')['amount'].agg(['sum', 'count']).reset_index()
            sales_summary.columns = ['id', 'total_purchases', 'num_purchases']
            # Объединяем с данными клиентов
            result: pd.DataFrame = clients.merge(sales_summary, on='id', how='left')
            result['total_purchases'] = result['total_purchases'].fillna(0)
            result['num_purchases'] = result['num_purchases'].fillna(0)
        else:
            result = clients
            result['total_purchases'] = 0
            result['num_purchases'] = 0

        return result

    def close(self) -> None:
        """
        Закрывает соединение с базой данных.

        Параметры:
            None

        Возвращает:
            None
        """
        if self.conn:
            self.conn.close()


class KPICalculator:
    """
    Класс для расчёта ключевых показателей эффективности (KPI).

    Методы:
        calculate_kpi(clients_df, sales_df) - расчёт всех показателей
        calculate_ltv(sales_df) - расчёт LTV
        calculate_avg_check(sales_df) - расчёт среднего чека
        calculate_conversion(clients_df, sales_df) - расчёт конверсии
    """

    @staticmethod
    def calculate_ltv(sales_df: pd.DataFrame) -> float:
        """
        Рассчитывает LTV (Lifetime Value) - пожизненную ценность клиента.

        Формула: LTV = Общая выручка / Количество клиентов

        Параметры:
            sales_df (pd.DataFrame): DataFrame с данными о продажах

        Возвращает:
            float: Значение LTV в рублях
        """
        if len(sales_df) == 0:
            return 0.0

        # Выручка по каждому клиенту
        revenue_by_client: pd.Series = sales_df.groupby('client_id')['amount'].sum()
        # Средняя выручка на клиента
        ltv: float = revenue_by_client.mean()
        return ltv

    @staticmethod
    def calculate_avg_check(sales_df: pd.DataFrame) -> float:
        """
        Рассчитывает средний чек.

        Формула: Средний чек = Общая выручка / Количество продаж

        Параметры:
            sales_df (pd.DataFrame): DataFrame с данными о продажах

        Возвращает:
            float: Значение среднего чека в рублях
        """
        if len(sales_df) == 0:
            return 0.0

        avg_check: float = sales_df['amount'].mean()
        return avg_check

    @staticmethod
    def calculate_conversion(clients_df: pd.DataFrame, sales_df: pd.DataFrame) -> float:
        """
        Рассчитывает конверсию - долю клиентов, совершивших покупку.

        Формула: Конверсия = (Клиенты с покупками / Всего клиентов) * 100%

        Параметры:
            clients_df (pd.DataFrame): DataFrame с данными о клиентах
            sales_df (pd.DataFrame): DataFrame с данными о продажах

        Возвращает:
            float: Значение конверсии в процентах
        """
        if len(clients_df) == 0:
            return 0.0

        if len(sales_df) == 0:
            return 0.0

        # Количество уникальных клиентов, совершивших покупки
        unique_buyers: int = sales_df['client_id'].nunique()
        # Общее количество клиентов
        total_clients: int = len(clients_df)
        # Конверсия в процентах
        conversion: float = (unique_buyers / total_clients) * 100
        return conversion

    @staticmethod
    def calculate_total_revenue(sales_df: pd.DataFrame) -> float:
        """
        Рассчитывает общую выручку.

        Формула: Общая выручка = Сумма всех продаж

        Параметры:
            sales_df (pd.DataFrame): DataFrame с данными о продажах

        Возвращает:
            float: Общая выручка в рублях
        """
        if len(sales_df) == 0:
            return 0.0

        total_revenue: float = sales_df['amount'].sum()
        return total_revenue

    @staticmethod
    def calculate_all_kpi(clients_df: pd.DataFrame, sales_df: pd.DataFrame) -> Dict[str, float]:
        """
        Рассчитывает все ключевые показатели эффективности.

        Параметры:
            clients_df (pd.DataFrame): DataFrame с данными о клиентах
            sales_df (pd.DataFrame): DataFrame с данными о продажах

        Возвращает:
            Dict[str, float]: Словарь со всеми KPI
        """
        kpi: Dict[str, float] = {
            'ltv': KPICalculator.calculate_ltv(sales_df),
            'avg_check': KPICalculator.calculate_avg_check(sales_df),
            'conversion': KPICalculator.calculate_conversion(clients_df, sales_df),
            'total_revenue': KPICalculator.calculate_total_revenue(sales_df),
            'total_clients': len(clients_df),
            'total_sales': len(sales_df),
            'cac': 500.0  # имитация стоимости привлечения клиента
        }
        return kpi


class CRMInterface:
    """
    Класс для управления пользовательским интерфейсом CRM-системы.

    Отвечает за:
        - Отображение меню
        - Обработку действий пользователя
        - Визуализацию данных
        - Создание форм для ввода данных

    Атрибуты:
        db (DatabaseManager): объект для работы с базой данных
        kpi_calculator (KPICalculator): объект для расчёта KPI
    """

    def __init__(self, db_manager: DatabaseManager) -> None:
        """
        Инициализация интерфейса CRM.

        Параметры:
            db_manager (DatabaseManager): объект для работы с базой данных

        Возвращает:
            None
        """
        self.db: DatabaseManager = db_manager
        self.kpi_calculator: KPICalculator = KPICalculator()

    def show_kpi_dashboard(self) -> None:
        """
        Отображает дашборд с ключевыми показателями эффективности.

        Показывает:
            - LTV (пожизненная ценность клиента)
            - Средний чек
            - Конверсию
            - Общую выручку
            - График продаж по категориям

        Параметры:
            None

        Возвращает:
            None
        """
        # Загружаем данные из базы
        clients_df: pd.DataFrame = self.db.get_clients()
        sales_df: pd.DataFrame = self.db.get_sales()

        # Рассчитываем KPI
        kpi: Dict[str, float] = self.kpi_calculator.calculate_all_kpi(clients_df, sales_df)

        # Выводим показатели
        print("=" * 50)
        print("КЛЮЧЕВЫЕ ПОКАЗАТЕЛИ ЭФФЕКТИВНОСТИ (KPI)")
        print("=" * 50)
        print(f"LTV (пожизненная ценность клиента): {kpi['ltv']:,.0f} руб.")
        print(f"Средний чек: {kpi['avg_check']:,.0f} руб.")
        print(f"Конверсия: {kpi['conversion']:.1f}%")
        print(f"Общая выручка: {kpi['total_revenue']:,.0f} руб.")
        print(f"Всего клиентов: {kpi['total_clients']:.0f}")
        print(f"Всего продаж: {kpi['total_sales']:.0f}")
        print(f"CAC (стоимость привлечения): {kpi['cac']:.0f} руб.")

        # Визуализация продаж по категориям
        if len(sales_df) > 0:
            print("\n" + "=" * 50)
            print("ВИЗУАЛИЗАЦИЯ ПРОДАЖ ПО КАТЕГОРИЯМ")
            print("=" * 50)

            # Группируем продажи по категориям
            sales_by_category: pd.DataFrame = sales_df.groupby('product_category')['amount'].sum().reset_index()

            # Создаём столбчатую диаграмму
            fig = px.bar(
                sales_by_category,
                x='product_category',
                y='amount',
                title='Выручка по категориям товаров',
                labels={'product_category': 'Категория', 'amount': 'Выручка (руб.)'},
                color='product_category'
            )
            fig.show()

    def show_client_list(self) -> None:
        """
        Отображает список всех клиентов в виде таблицы.

        Параметры:
            None

        Возвращает:
            None
        """
        print("=" * 50)
        print("СПИСОК КЛИЕНТОВ")
        print("=" * 50)

        clients_with_sales: pd.DataFrame = self.db.get_clients_with_sales()

        # Выбираем нужные колонки для отображения
        display_columns: List[str] = ['id', 'name', 'phone', 'segment', 'total_purchases', 'num_purchases']
        display(clients_with_sales[display_columns])

    def create_add_client_form(self) -> widgets.VBox:
        """
        Создаёт форму для добавления нового клиента.

        Параметры:
            None

        Возвращает:
            widgets.VBox: Вертикальный контейнер с элементами формы
        """
        # Поле для ввода ФИО
        name_input = widgets.Text(
            description='ФИО:',
            placeholder='Введите фамилию, имя, отчество',
            style={'description_width': 'initial'}
        )

        # Поле для ввода телефона
        phone_input = widgets.Text(
            description='Телефон:',
            placeholder='+7-XXX-XXX-XX-XX',
            style={'description_width': 'initial'}
        )

        # Поле для ввода email
        email_input = widgets.Text(
            description='Email:',
            placeholder='example@mail.ru',
            style={'description_width': 'initial'}
        )

        # Выпадающий список для выбора сегмента
        segment_input = widgets.Dropdown(
            options=['VIP', 'Обычный', 'Новый'],
            description='Сегмент:',
            style={'description_width': 'initial'}
        )

        # Кнопка для отправки формы
        submit_button = widgets.Button(
            description='Добавить клиента',
            button_style='success'
        )

        # Область для вывода результата
        output_area = widgets.Output()

        # Обработчик нажатия кнопки
        def on_submit_click(b: widgets.Button) -> None:
            with output_area:
                clear_output()  # очищаем предыдущий вывод

                # Проверяем, что ФИО введено
                if not name_input.value:
                    print("Ошибка: Введите ФИО клиента")
                    return

                # Добавляем клиента в базу данных
                success: bool = self.db.add_client(
                    name=name_input.value,
                    phone=phone_input.value,
                    email=email_input.value,
                    segment=segment_input.value
                )

                if success:
                    print(f"Клиент '{name_input.value}' успешно добавлен!")
                    # Очищаем поля формы
                    name_input.value = ''
                    phone_input.value = ''
                    email_input.value = ''
                else:
                    print("Ошибка при добавлении клиента")

        submit_button.on_click(on_submit_click)

        # Собираем все элементы в вертикальный контейнер
        form = widgets.VBox([
            name_input,
            phone_input,
            email_input,
            segment_input,
            submit_button,
            output_area
        ])

        return form

    def create_add_sale_form(self) -> widgets.VBox:
        """
        Создаёт форму для добавления новой продажи.

        Параметры:
            None

        Возвращает:
            widgets.VBox: Вертикальный контейнер с элементами формы
        """
        # Получаем список клиентов для выпадающего списка
        clients_df: pd.DataFrame = self.db.get_clients()

        if len(clients_df) == 0:
            # Если нет клиентов, показываем сообщение
            return widgets.HTML("<b>Сначала добавьте клиентов в системе</b>")

        # Создаём словарь {имя клиента: id клиента}
        client_options: Dict[str, int] = {row['name']: row['id'] for _, row in clients_df.iterrows()}

        # Выпадающий список для выбора клиента
        client_select = widgets.Dropdown(
            options=list(client_options.keys()),
            description='Клиент:',
            style={'description_width': 'initial'}
        )

        # Поле для ввода суммы
        amount_input = widgets.IntText(
            description='Сумма (руб.):',
            value=1000,
            style={'description_width': 'initial'}
        )

        # Выпадающий список для выбора категории
        category_select = widgets.Dropdown(
            options=['Электроника', 'Одежда', 'Продукты', 'Косметика', 'Дом'],
            description='Категория:',
            style={'description_width': 'initial'}
        )

        # Кнопка для отправки формы
        submit_button = widgets.Button(
            description='Добавить продажу',
            button_style='success'
        )

        # Область для вывода результата
        output_area = widgets.Output()

        # Обработчик нажатия кнопки
        def on_submit_click(b: widgets.Button) -> None:
            with output_area:
                clear_output()  # очищаем предыдущий вывод

                # Получаем ID выбранного клиента
                client_id: int = client_options[client_select.value]

                # Добавляем продажу в базу данных
                success: bool = self.db.add_sale(
                    client_id=client_id,
                    amount=float(amount_input.value),
                    category=category_select.value
                )

                if success:
                    print(f"Продажа добавлена! Клиент: {client_select.value}, сумма: {amount_input.value} руб.")
                    # Сбрасываем сумму на значение по умолчанию
                    amount_input.value = 1000
                else:
                    print("Ошибка при добавлении продажи")

        submit_button.on_click(on_submit_click)

        # Собираем все элементы в вертикальный контейнер
        form = widgets.VBox([
            client_select,
            amount_input,
            category_select,
            submit_button,
            output_area
        ])

        return form

    def run(self) -> None:
        """
        Запускает основной цикл работы CRM-системы.

        Создаёт главное меню и обрабатывает выбор пользователя.

        Параметры:
            None

        Возвращает:
            None
        """
        # Выводим заголовок
        print("=" * 60)
        print("CRM СИСТЕМА ДЛЯ ОПТИМИЗАЦИИ ПРОДАЖ В РИТЕЙЛЕ")
        print("=" * 60)
        print("Версия 2.0")
        print("Разработано в рамках НИР")
        print("=" * 60)

        # Создаём главное меню
        menu = widgets.Dropdown(
            options=[
                'Показать KPI (дашборд)',
                'Список клиентов',
                'Добавить клиента',
                'Добавить продажу'
            ],
            description='Выберите действие:',
            style={'description_width': 'initial'}
        )

        # Область для отображения содержимого
        content_area = widgets.Output()

        # Функция обработки выбора меню
        def on_menu_change(change: Dict[str, Any]) -> None:
            with content_area:
                clear_output(wait=True)  # очищаем предыдущее содержимое

                selected: str = change['new']

                if selected == 'Показать KPI (дашборд)':
                    self.show_kpi_dashboard()
                elif selected == 'Список клиентов':
                    self.show_client_list()
                elif selected == 'Добавить клиента':
                    form = self.create_add_client_form()
                    display(form)
                elif selected == 'Добавить продажу':
                    form = self.create_add_sale_form()
                    display(form)

        # Подписываемся на изменения меню
        menu.observe(on_menu_change, names='value')

        # Отображаем меню и область для содержимого
        display(menu)
        display(content_area)

        # Запускаем начальное отображение (показываем KPI по умолчанию)
        on_menu_change({'new': 'Показать KPI (дашборд)'})


# Главная функция запуска приложения
def main() -> None:
    """
    Главная функция для запуска CRM-системы.

    Создаёт экземпляры классов и запускает интерфейс.

    Параметры:
        None

    Возвращает:
        None
    """
    # Создаём менеджер базы данных
    db_manager: DatabaseManager = DatabaseManager()

    # Создаём интерфейс CRM
    app: CRMInterface = CRMInterface(db_manager)

    # Запускаем приложение
    app.run()


# Точка входа в программу
if __name__ == "__main__":
    main()

CRM СИСТЕМА ДЛЯ ОПТИМИЗАЦИИ ПРОДАЖ В РИТЕЙЛЕ
Версия 2.0
Разработано в рамках НИР


Dropdown(description='Выберите действие:', options=('Показать KPI (дашборд)', 'Список клиентов', 'Добавить кли…

Output()